# Fine-tuned Phi-2 Inference & Evaluation (Colab, T4 GPU)

**Project**: Cybersecurity Domain RAG Pipeline — Final Year Project (CB013309)
**Scope**: inference + evaluation only for the fine-tuned (Phi-2 + QLoRA) side of the pipeline.

This is **separate from `finetune_cybersecurity.ipynb`** (the training notebook). It does not
train anything — it loads the already-trained QLoRA adapter, runs the same
retrieval-augmented pipeline used in `src/rag_pipeline.py`, and evaluates all 45 held-out
questions, writing results to a CSV on Drive.

**Requires a GPU runtime**: `Runtime > Change runtime type > T4 GPU`.

**Output schema** (long format, one row per question):
`question_id, system, answer, confidence_score, support_rate, context_truncated`
(`system` is always `"finetuned"` in this notebook — the baseline/Llama run happens
separately via `run_evaluation.py`.)

**Resumable by design**: every row is appended to the output CSV immediately after that
question completes (no in-memory buffering), and the loop skips any `question_id` already
present in the CSV. If Colab disconnects or times out, just re-run the loop cell — it picks
up where it left off.

In [ ]:
!pip uninstall -y torchao -q

!pip install -q \
    transformers \
    peft \
    accelerate \
    bitsandbytes \
    langchain \
    langchain-classic \
    langchain-community \
    langchain-core \
    langchain-text-splitters \
    langchain-experimental \
    langchain-huggingface \
    langchain-groq \
    chromadb \
    sentence-transformers \
    python-dotenv

import torch

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: no GPU detected — go to Runtime > Change runtime type > T4 GPU.")

## Step 1 — Mount Google Drive and configure paths

Adjust these if your Drive layout differs:

- `ADAPTER_DIR_DRIVE` — where the trained QLoRA adapter lives (`FYP/output/lora_adapter`,
  per project convention). If it isn't there, this notebook falls back to the adapter
  copy committed in the repo itself (`models/lora_adapter`).
- `OUTPUT_CSV` — results are appended here **incrementally, one question at a time**, so a
  Colab disconnect never loses progress.

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

# ── Configure these to match your Drive layout ─────────────────────────────
ADAPTER_DIR_DRIVE = "/content/drive/MyDrive/FYP/output/lora_adapter"
OUTPUT_CSV        = "/content/drive/MyDrive/FYP/output/finetuned_eval_results_long.csv"
REPO_GIT_URL = "https://github.com/ZuhriAshroff/cybersecurity-rag-llm-fyp-v2.git"
REPO_DIR          = "/content/fyp_v2"
# ────────────────────────────────────────────────────────────────────────────

os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
print("Adapter (Drive):", ADAPTER_DIR_DRIVE, "exists:", os.path.isdir(ADAPTER_DIR_DRIVE))
print("Output CSV     :", OUTPUT_CSV)

## Step 2 — Sync the repo (retrieval pipeline, corpus, held-out questions)

Clones (or pulls, if already present) the project repo into the Colab VM so we can reuse
`src/rag_pipeline.py` — the semantic-chunking retriever, cross-encoder reranker, and the
**fixed** `query_finetuned()` (context-only left-truncation, with a reserved generation-token
budget) — directly, instead of reimplementing it.

The corpus and held-out questions are data, not code, so this clone is a hard prerequisite —
if it fails (e.g. the repo is private and Colab has no credentials for it), grant Colab
access or upload the repo another way before continuing.

If the repo's `rag_pipeline.py` doesn't yet contain the fix (e.g. it hasn't been pushed from
local yet), this notebook detects that automatically and falls back to an inline copy of the
fixed logic (Step 2b), so the notebook produces correct results either way.

In [ ]:
import subprocess, sys, os

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_GIT_URL, REPO_DIR], check=True)

sys.path.insert(0, os.path.join(REPO_DIR, "src"))

USE_INLINE = False
try:
    import rag_pipeline as rp
    if not hasattr(rp, "_real_max_positions"):
        print("rag_pipeline.py in the synced repo predates the dynamic token-budget fix — using inline fallback.")
        USE_INLINE = True
    else:
        print("Imported rag_pipeline.py from the synced repo — using it directly (fix present).")
except Exception as e:
    print(f"Could not import rag_pipeline.py ({e}) — using inline fallback.")
    USE_INLINE = True

# Point the fine-tuned model loader at the Drive adapter if it's there, else fall
# back to the copy committed in the repo itself.
if not USE_INLINE:
    if os.path.isdir(ADAPTER_DIR_DRIVE):
        rp.ADAPTER_PATH = ADAPTER_DIR_DRIVE
    else:
        print(f"'{ADAPTER_DIR_DRIVE}' not found — falling back to the repo's bundled adapter "
              f"at {rp.ADAPTER_PATH}.")

## Step 2b — Inline fallback (only runs if `USE_INLINE` is `True`)

Self-contained copy of the retrieval pipeline and the fixed `query_finetuned()`, mirroring
`src/rag_pipeline.py`: the real context limit is read from `model.config.max_position_embeddings`
(not assumed), the instruction/question template is tokenized first, `GENERATION_TOKEN_BUDGET`
(256) and a small margin are reserved, and only the **context** is truncated
(`truncation_side="left"`) to whatever's left — the question is never touched. A final check
retokenizes the assembled prompt and trims further if needed before generation, so the prompt +
generation budget can never exceed the model's real context window. Skipped entirely when the
synced repo already has this fix.

In [ ]:
if USE_INLINE:
    import re as _re
    import numpy as np
    import torch
    from typing import Any, List
    from langchain_community.document_loaders import TextLoader
    from langchain_huggingface import HuggingFaceEmbeddings
    from langchain_community.vectorstores import Chroma
    from langchain_core.retrievers import BaseRetriever
    from langchain_core.callbacks.manager import CallbackManagerForRetrieverRun
    from langchain_experimental.text_splitter import SemanticChunker
    from sentence_transformers import CrossEncoder
    from pydantic import Field
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from peft import PeftModel

    CORPUS_DIR   = os.path.join(REPO_DIR, "corpus")
    ADAPTER_PATH = ADAPTER_DIR_DRIVE if os.path.isdir(ADAPTER_DIR_DRIVE) else os.path.join(REPO_DIR, "models", "lora_adapter")
    PHI2_BASE    = "microsoft/phi-2"
    INITIAL_K    = 10
    TOP_K        = 3
    SUPPORT_THRESHOLD = 0.35
    PHI2_MAX_POSITIONS_FALLBACK = 2048  # used only if introspection below fails
    CONTEXT_TRUNCATION_MARGIN = 50
    GENERATION_TOKEN_BUDGET = 256

    _ft_model_cache: dict = {}

    def _real_max_positions(model, tokenizer) -> int:
        """Phi-2's actual context window, read from the loaded model/tokenizer rather
        than assumed, so this never silently drifts from whatever checkpoint is loaded."""
        max_positions = getattr(model.config, "max_position_embeddings", None)
        if not max_positions:
            max_positions = getattr(tokenizer, "model_max_length", None)
        if not max_positions or max_positions > 1_000_000:  # unset sentinel guard
            max_positions = PHI2_MAX_POSITIONS_FALLBACK
        return max_positions

    class ParentChildRetriever(BaseRetriever):
        vectorstore:  Any = Field(...)
        parent_store: Any = Field(...)
        initial_k:    int = INITIAL_K
        model_config = {"arbitrary_types_allowed": True}

        def _get_relevant_documents(self, query, *, run_manager: CallbackManagerForRetrieverRun):
            child_hits = self.vectorstore.similarity_search(query, k=self.initial_k)
            seen, parents = set(), []
            for chunk in child_hits:
                src = chunk.metadata.get("source", "unknown")
                if src not in seen:
                    seen.add(src)
                    parents.append(self.parent_store.get(src, chunk))
            return parents

    class RerankedRetriever(BaseRetriever):
        base_retriever: Any = Field(...)
        cross_encoder:  Any = Field(...)
        final_k:        int = TOP_K
        model_config = {"arbitrary_types_allowed": True}

        def _get_relevant_documents(self, query, *, run_manager: CallbackManagerForRetrieverRun):
            candidates = self.base_retriever.invoke(query)
            if not candidates:
                return []
            pairs  = [[query, doc.page_content] for doc in candidates]
            scores = self.cross_encoder.predict(pairs)
            ranked = sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)
            return [doc for _, doc in ranked[:self.final_k]]

    def load_corpus():
        import glob
        docs = []
        txt_files = sorted(glob.glob(os.path.join(CORPUS_DIR, "*.txt")))
        if not txt_files:
            raise FileNotFoundError(f"No .txt files found in '{CORPUS_DIR}/'.")
        for path in txt_files:
            loaded = TextLoader(path, encoding="utf-8").load()
            for doc in loaded:
                doc.metadata["source"] = os.path.basename(path)
            docs.extend(loaded)
        print(f"Loaded {len(docs)} document(s) from '{CORPUS_DIR}/'")
        return docs

    def build_retriever(docs):
        embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
        child_chunks = SemanticChunker(embeddings).split_documents(docs)
        print(f"Created {len(child_chunks)} semantic child chunks from {len(docs)} document(s).")
        parent_store = {}
        for doc in docs:
            src = doc.metadata.get("source", "unknown")
            if src not in parent_store:
                parent_store[src] = doc
        vectorstore = Chroma.from_documents(child_chunks, embeddings)
        base_retriever = ParentChildRetriever(vectorstore=vectorstore, parent_store=parent_store)
        cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
        retriever = RerankedRetriever(base_retriever=base_retriever, cross_encoder=cross_encoder)
        print(f"Ready — {INITIAL_K} candidates -> rerank -> top {TOP_K} to LLM.")
        return retriever, embeddings

    def score_hallucination(answer, source_docs, embeddings):
        sentences = [s.strip() for s in _re.split(r'(?<=[.!?])\s+', answer) if len(s.strip()) > 15]
        if not sentences:
            return 100.0, []
        context_texts = [doc.page_content for doc in source_docs]
        sent_vecs = np.array(embeddings.embed_documents(sentences))
        ctx_vecs  = np.array(embeddings.embed_documents(context_texts))
        sent_vecs /= np.linalg.norm(sent_vecs, axis=1, keepdims=True)
        ctx_vecs  /= np.linalg.norm(ctx_vecs,  axis=1, keepdims=True)
        sim_matrix = sent_vecs @ ctx_vecs.T
        max_sims   = sim_matrix.max(axis=1)
        results    = [(sent, float(sim), float(sim) >= SUPPORT_THRESHOLD) for sent, sim in zip(sentences, max_sims)]
        confidence = float(np.mean(max_sims)) * 100
        return confidence, results

    def load_finetuned_model():
        if "model" in _ft_model_cache:
            return _ft_model_cache["model"], _ft_model_cache["tokenizer"]
        if not os.path.isdir(ADAPTER_PATH):
            raise FileNotFoundError(f"LoRA adapter not found at '{ADAPTER_PATH}'.")

        tokenizer = AutoTokenizer.from_pretrained(PHI2_BASE, trust_remote_code=True)

        if torch.cuda.is_available():
            print(f"Loading {PHI2_BASE} in float16 on GPU (device_map='auto')...")
            base_model = AutoModelForCausalLM.from_pretrained(
                PHI2_BASE, torch_dtype=torch.float16, trust_remote_code=True,
                device_map="auto",
            )
        else:
            print(f"Loading {PHI2_BASE} in float16 on CPU...")
            base_model = AutoModelForCausalLM.from_pretrained(
                PHI2_BASE, torch_dtype=torch.float16, trust_remote_code=True,
            )
            base_model.to("cpu")

        print(f"Attaching LoRA adapter from '{ADAPTER_PATH}'...")
        model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
        model.eval()
        _ft_model_cache["model"] = model
        _ft_model_cache["tokenizer"] = tokenizer
        print(f"Fine-tuned model ready on device: {next(model.parameters()).device}")
        return model, tokenizer

    def query_finetuned(question, retriever, embeddings):
        model, tokenizer = load_finetuned_model()
        source_docs = retriever.invoke(question)
        context = "\n\n".join(doc.page_content for doc in source_docs)

        header = (
            "You are a cybersecurity domain expert.\n"
            "Using the retrieved context below, give a clear, synthesised answer drawn from the available sources.\n"
            "Always provide a concrete, best-effort answer — never refuse to answer.\n"
            "If the context only partially covers the question, answer what you can and note the gap briefly.\n\n"
            "Context:\n"
        )
        footer = f"\n\nQuestion: {question}\n\nAnswer:"

        real_max_positions = _real_max_positions(model, tokenizer)

        template_token_count = len(tokenizer(header + footer)["input_ids"])
        available_context_tokens = max(
            0,
            real_max_positions - template_token_count - GENERATION_TOKEN_BUDGET - CONTEXT_TRUNCATION_MARGIN,
        )

        # verbose=False: this intentionally tokenizes the raw, untruncated context to
        # measure how much would need trimming — it's expected to exceed the model's
        # max length for long retrievals, so the library's generic overflow warning
        # would be misleading noise here. We log our own warning below instead, only
        # when truncation actually happens.
        original_context_tokens = len(tokenizer(context, verbose=False)["input_ids"])

        prior_side = tokenizer.truncation_side
        tokenizer.truncation_side = "left"
        try:
            truncated_ids = tokenizer(context, truncation=True, max_length=available_context_tokens)["input_ids"]
        finally:
            tokenizer.truncation_side = prior_side

        used_context_tokens = len(truncated_ids)
        context_truncated = used_context_tokens < original_context_tokens
        trimmed_context = tokenizer.decode(truncated_ids, skip_special_tokens=True)

        if context_truncated:
            print(
                f"WARNING: context truncated to fit the token budget — would have been "
                f"{template_token_count + original_context_tokens} tokens (over the "
                f"{real_max_positions}-token limit); trimmed context from "
                f"{original_context_tokens} to {used_context_tokens} tokens."
            )

        prompt = header + trimmed_context + footer

        # Hard safety net: retokenizing the reassembled string can drift a few tokens
        # from the token-id-level budget above (BPE merges differ at the splice
        # points). If that ever pushes the prompt + generation budget over the
        # model's real limit, trim the context a little further and retry, rather
        # than silently handing the model an over-length sequence.
        for _ in range(3):
            inputs = tokenizer(prompt, return_tensors="pt", verbose=False)
            final_prompt_tokens = inputs["input_ids"].shape[-1]
            if final_prompt_tokens + GENERATION_TOKEN_BUDGET <= real_max_positions:
                break
            overflow = final_prompt_tokens + GENERATION_TOKEN_BUDGET - real_max_positions
            print(
                f"WARNING: reassembled prompt drifted {overflow} token(s) over budget "
                f"({final_prompt_tokens} prompt + {GENERATION_TOKEN_BUDGET} generation > "
                f"{real_max_positions}) — trimming context further before generation."
            )
            tokenizer.truncation_side = "left"
            truncated_ids = tokenizer(
                trimmed_context, truncation=True,
                max_length=max(0, len(truncated_ids) - overflow - 5),
            )["input_ids"]
            tokenizer.truncation_side = prior_side
            used_context_tokens = len(truncated_ids)
            context_truncated = True
            trimmed_context = tokenizer.decode(truncated_ids, skip_special_tokens=True)
            prompt = header + trimmed_context + footer
        else:
            final_prompt_tokens = tokenizer(prompt, return_tensors="pt", verbose=False)["input_ids"].shape[-1]

        assert final_prompt_tokens + GENERATION_TOKEN_BUDGET <= real_max_positions, (
            f"Prompt ({final_prompt_tokens} tokens) + generation budget "
            f"({GENERATION_TOKEN_BUDGET}) still exceeds the model's real context window "
            f"({real_max_positions}) after correction — this should never happen."
        )

        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.no_grad():
            output_ids = model.generate(
                **inputs, max_new_tokens=GENERATION_TOKEN_BUDGET, do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
        answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

        seen, sources = set(), []
        for doc in source_docs:
            src = doc.metadata.get("source", "unknown")
            if src not in seen:
                seen.add(src)
                sources.append((src, doc.page_content[:300]))

        confidence, sentence_scores = score_hallucination(answer, source_docs, embeddings)
        n_supported = sum(1 for _, _, ok in sentence_scores if ok)

        return {
            "answer": answer, "sources": sources, "source_docs": source_docs,
            "confidence": confidence, "sentence_scores": sentence_scores, "n_supported": n_supported,
            "context_truncated": context_truncated,
            "original_context_tokens": original_context_tokens,
            "used_context_tokens": used_context_tokens,
        }

    print("Inline fallback pipeline defined.")
else:
    print("Skipping inline fallback — using imported rag_pipeline.py.")

## Step 3 — Unify function names

So the rest of the notebook doesn't care whether Step 2 or Step 2b supplied the pipeline.

In [ ]:
if USE_INLINE:
    _load_corpus           = load_corpus
    _build_retriever       = build_retriever
    _load_finetuned_model  = load_finetuned_model
    _query_finetuned       = query_finetuned
else:
    _load_corpus           = rp.load_corpus
    _build_retriever       = rp.build_retriever
    _load_finetuned_model  = rp.load_finetuned_model
    _query_finetuned       = rp.query_finetuned

print("Pipeline source:", "inline fallback" if USE_INLINE else "imported rag_pipeline.py")

## Step 4 — Build the corpus retriever (one-time; re-embeds the corpus)

In [ ]:
docs = _load_corpus()
retriever, embeddings = _build_retriever(docs)

## Step 5 — Load the fine-tuned model (base Phi-2 + QLoRA adapter)

Loaded once and cached — the loop below reuses this instance for all 45 questions.

In [ ]:
model, _ = _load_finetuned_model()
resolved_device = next(model.parameters()).device
print(f"Fine-tuned model loaded and cached. Resolved device: {resolved_device}")

if torch.cuda.is_available():
    assert resolved_device.type == "cuda", (
        f"CUDA is available but the model resolved to '{resolved_device}' instead of cuda — "
        "device placement did not take effect as expected."
    )
    print("Confirmed: model is on GPU (cuda:0).")
else:
    print("No GPU detected in this runtime — model is on CPU. "
          "Go to Runtime > Change runtime type > T4 GPU and re-run if you expected a GPU.")

## Step 6 — Load the 45 held-out evaluation questions

Same split as `run_evaluation.py`: the **last 20%** of `training_data/cybersecurity_qa.jsonl`
(45 of 225 rows), so `question_id` numbering matches the baseline run exactly.

In [ ]:
import json, re as _re2

QA_FILE = os.path.join(REPO_DIR, "training_data", "cybersecurity_qa.jsonl")
_QUESTION_PATTERN = _re2.compile(r"Question:\s*(.+?)\s*\nContext:", _re2.DOTALL)

def extract_question(row: dict) -> str:
    if row.get("question"):
        return row["question"].strip()
    m = _QUESTION_PATTERN.search(row.get("prompt", ""))
    return m.group(1).strip() if m else row.get("prompt", "").strip()

rows = []
with open(QA_FILE, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))

split_idx = int(len(rows) * 0.8)
held_out = rows[split_idx:]
print(f"Held-out test split: last {len(held_out)} of {len(rows)} total rows.")

questions = [
    (i, extract_question(row)) for i, row in enumerate(held_out, 1) if extract_question(row)
]
print(f"{len(questions)} question(s) ready for evaluation.")

## Step 7 — Run the evaluation loop (resumable)

Appends one row per question directly to `OUTPUT_CSV` on Drive as soon as it's computed — no
in-memory buffering — so a disconnect never loses completed work. Re-running this cell skips
any `question_id` already present in the CSV.

Columns: `question_id, system, answer, confidence_score, support_rate, context_truncated`.

In [ ]:
import csv

FIELDNAMES = ["question_id", "system", "answer", "confidence_score", "support_rate", "context_truncated"]

def load_completed_ids(path):
    if not os.path.isfile(path):
        return set()
    with open(path, "r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        return {int(row["question_id"]) for row in reader if row.get("system") == "finetuned"}

completed = load_completed_ids(OUTPUT_CSV)
print(f"{len(completed)}/{len(questions)} already completed - resuming.")

file_exists = os.path.isfile(OUTPUT_CSV)
with open(OUTPUT_CSV, "a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
    if not file_exists:
        writer.writeheader()
        f.flush()
        os.fsync(f.fileno())

    for qid, question in questions:
        if qid in completed:
            print(f"[{qid}/{len(questions)}] already done - skipping")
            continue

        print(f"[{qid}/{len(questions)}] {question[:80]}")
        result = _query_finetuned(question, retriever, embeddings)

        n_sentences = len(result["sentence_scores"])
        support_rate = (result["n_supported"] / n_sentences) if n_sentences else 1.0

        writer.writerow({
            "question_id":       qid,
            "system":            "finetuned",
            "answer":            result["answer"],
            "confidence_score":  round(result["confidence"], 2),
            "support_rate":      round(support_rate, 4),
            "context_truncated": result["context_truncated"],
        })
        f.flush()
        os.fsync(f.fileno())

        print(f"  confidence={result['confidence']:.1f}%  support_rate={support_rate:.2f}  "
              f"context_truncated={result['context_truncated']}")
        print(f"{qid}/{len(questions)} done")

print("Evaluation loop finished (or all questions were already complete).")

## Step 8 — Verify results

In [ ]:
import pandas as pd

results = pd.read_csv(OUTPUT_CSV)
print(f"Rows: {len(results)} (expected {len(questions)})")
print(f"context_truncated=True: {int(results['context_truncated'].sum())} / {len(results)}")
results.tail()